In [1]:
import numpy as np

# --- constants ---
g = 9.81
L1 = L2 = 1.0
m1 = m2 = 1.0

def wrap_pi(x):
    return (x + np.pi) % (2*np.pi) - np.pi

def f(state):
    # Differentiate
    # state = [theta1, omega1, theta2, omega2]
    th1, w1, th2, w2 = state
    d = th1 - th2
    denom = (2*m1 + m2 - m2*np.cos(2*d))

    dth1 = w1
    dth2 = w2

    dw1 = (-g*(2*m1+m2)*np.sin(th1)
           - m2*g*np.sin(th1 - 2*th2)
           - 2*np.sin(d)*m2*(w2**2*L2 + w1**2*L1*np.cos(d))) / (L1*denom)

    dw2 = (2*np.sin(d) * (w1**2*L1*(m1+m2)
           + g*(m1+m2)*np.cos(th1)
           + w2**2*L2*m2*np.cos(d))) / (L2*denom)

    return np.array([dth1, dw1, dth2, dw2], dtype=np.float64)

def rk4_step(state, dt):
    k1 = dt * f(state)
    k2 = dt * f(state + 0.5*k1)
    k3 = dt * f(state + 0.5*k2)
    k4 = dt * f(state + k3)
    return state + (k1 + 2*k2 + 2*k3 + k4) / 6.0

def simulate(theta1_0, theta2_0, dt=0.005, T=10.0):
    n = int(T/dt) + 1
    traj = np.zeros((n, 4), dtype=np.float64)  # [theta1, theta2, omega1, omega2]
    state = np.array([theta1_0, 0.0, theta2_0, 0.0], dtype=np.float64)

    for i in range(n):
        traj[i, 0] = wrap_pi(state[0])
        traj[i, 1] = wrap_pi(state[2])
        traj[i, 2] = state[1]
        traj[i, 3] = state[3]
        if i < n - 1:
            state = rk4_step(state, dt)

    return traj

# --- generate data for different initial conditions ---
np.random.seed(0)

n_traj = 50
dt = 0.005
T = 10.0

# sample initial angles (radians); velocities start at 0
theta1s = np.random.uniform(np.pi/2, 3*np.pi/4, size=n_traj)
theta2s = np.random.uniform(np.pi/2, 3*np.pi/4, size=n_traj)

all_traj = np.zeros((n_traj, int(T/dt)+1, 4), dtype=np.float64)

for i in range(n_traj):
    all_traj[i] = simulate(theta1s[i], theta2s[i], dt=dt, T=T)

# Save:
# all_traj shape: (50, 2001, 4)  -> [theta1, theta2, omega1, omega2]
# ics shape: (50, 2)             -> [theta1_0, theta2_0]
np.save("double_pendulum_trajs.npy", all_traj)
np.save("double_pendulum_ics.npy", np.stack([theta1s, theta2s], axis=1))

print("Saved:")
print("  double_pendulum_trajs.npy", all_traj.shape)
print("  double_pendulum_ics.npy", (n_traj, 2))

Saved:
  double_pendulum_trajs.npy (50, 2001, 4)
  double_pendulum_ics.npy (50, 2)


In [6]:
all_traj[2] 

array([[ 2.04420558,  2.34706332,  0.        ,  0.        ],
       [ 2.04408182,  2.34709395, -0.04950163,  0.01224825],
       [ 2.04371056,  2.34718579, -0.09900341,  0.02448687],
       ...,
       [-2.61782647, -0.58874333,  1.18535126,  3.90652036],
       [-2.61172305, -0.56908077,  1.25621819,  3.95848039],
       [-2.60526226, -0.54915879,  1.3283018 ,  4.0102843 ]],
      shape=(2001, 4))

In [8]:
ics = np.load("double_pendulum_ics.npy")
ics

array([[2.00183344, 2.01862782],
       [2.13250474, 1.91527315],
       [2.04420558, 2.34706332],
       [1.99874658, 1.65094213],
       [1.90353403, 1.73484775],
       [2.07808038, 1.69748853],
       [1.91447652, 2.08374641],
       [2.2711932 , 1.76973109],
       [2.32765529, 1.93703595],
       [1.87195059, 1.76276774],
       [2.19261572, 1.69565075],
       [1.98618943, 1.65748476],
       [2.01693748, 2.08627638],
       [2.29775823, 1.67932496],
       [1.62658792, 1.72519175],
       [1.63922752, 1.8603924 ],
       [1.58667582, 2.2156029 ],
       [2.22473442, 1.64705949],
       [2.18195921, 2.22891672],
       [2.25410227, 1.64627184],
       [2.33940138, 2.3377058 ],
       [2.198454  , 1.93887412],
       [1.93324137, 2.33794269],
       [2.18382251, 2.04584089],
       [1.66368884, 2.15141258],
       [2.07338912, 1.60157435],
       [1.68338574, 1.7929124 ],
       [2.31273756, 1.66519849],
       [1.98065504, 1.80338429],
       [1.89647105, 1.66404486],
       [1.